# 04 - Silver transactions (typed, validated, reference-checked, deduplicated)

**Inputs:** `bronze_transactions` and the accepted `silver_orders` (created by `03_silver_orders`)

**Outputs (every Bronze row lands in exactly one):**
| Table | Contains |
|---|---|
| `silver_transactions` | valid rows that **won** deduplication (one row per `txn_id`) |
| `quarantine_transactions` | rows that failed >= 1 rule (including orphan `order_id`), with `rejection_reason` |
| `duplicate_losers_transactions` | valid rows that **lost** to another valid row with the same `txn_id` |

### Precedence (copy into README)
**Validate first, then deduplicate among valid records**, same as orders.
The order reference is part of validation, so an orphan transaction is quarantined and never competes.

### Order-reference rule (copy into README)
A transaction's `order_id` is valid **only if that `order_id` exists in the accepted `silver_orders`**.
- An order that was **quarantined** (and has no accepted row with that id) -> its transactions are quarantined.
- An order id that has duplicate rows is valid as long as one row won: the id exists in `silver_orders`.
  The rows in `duplicate_losers_orders` share the winner's `order_id`, so transactions are linked to the id, not to a specific loser row.

### Rerun behavior
Full rebuild with `CREATE OR REPLACE TABLE`; reruns never append twice.

### Important
Allowed lists and limits are **rules you choose**, not assignment-provided rules. Use the pre-flight (section 3) to set them from your real data.

## 1. Parameters (these ARE your documented validation rules)

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")

# PLACEHOLDER defaults - confirm them in section 3 before trusting the results.
dbutils.widgets.text("allowed_payment_methods", "credit_card,debit_card,paypal,bank_transfer", "Allowed payment methods (lowercase, comma list)")
dbutils.widgets.text("allowed_currencies", "USD,EUR,GBP", "Allowed currencies (UPPERCASE, comma list)")
dbutils.widgets.text("allowed_statuses", "success,failed,pending,refunded", "Allowed txn statuses (lowercase, comma list)")

dbutils.widgets.text("min_amount", "0.01", "Min amount (inclusive)")
dbutils.widgets.text("max_amount", "1000000", "Max amount (inclusive)")
dbutils.widgets.text("min_txn_date", "2000-01-01", "Earliest acceptable txn_ts date (yyyy-MM-dd)")

In [0]:
import datetime
from decimal import Decimal

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()


def parse_list(text, transform):
    """'a, B ,c' -> ['a','b','c'] using the given transform; blank text -> []"""
    return [transform(x.strip()) for x in text.split(",") if x.strip()]


def sql_in_list(values):
    """['a','b'] -> 'a', 'b' (quoted for a SQL IN (...) list, quotes escaped)"""
    return ", ".join("'" + v.replace("'", "''") + "'" for v in values)


ALLOWED_PAYMENT_METHODS = parse_list(dbutils.widgets.get("allowed_payment_methods"), str.lower)
ALLOWED_CURRENCIES = parse_list(dbutils.widgets.get("allowed_currencies"), str.upper)
ALLOWED_STATUSES = parse_list(dbutils.widgets.get("allowed_statuses"), str.lower)

MIN_AMOUNT = Decimal(dbutils.widgets.get("min_amount").strip())
MAX_AMOUNT = Decimal(dbutils.widgets.get("max_amount").strip())
MIN_TXN_DATE = datetime.date.fromisoformat(dbutils.widgets.get("min_txn_date").strip())

# "Now" is captured ONCE so the future-timestamp rule is identical for every table written in this run
RUN_TS = spark.sql("SELECT CAST(current_timestamp() AS STRING) AS t").collect()[0]["t"]
SESSION_TZ = spark.conf.get("spark.sql.session.timeZone")

assert ALLOWED_PAYMENT_METHODS, "allowed_payment_methods cannot be empty (guide requires a supported set)."
assert ALLOWED_CURRENCIES, "allowed_currencies cannot be empty (guide requires a documented list)."
assert ALLOWED_STATUSES, "allowed_statuses cannot be empty (guide requires a documented set)."

BRONZE = f"`{CATALOG}`.`{SCHEMA}`.`bronze_transactions`"
SILVER_ORDERS = f"`{CATALOG}`.`{SCHEMA}`.`silver_orders`"
SILVER = f"`{CATALOG}`.`{SCHEMA}`.`silver_transactions`"
QUARANTINE = f"`{CATALOG}`.`{SCHEMA}`.`quarantine_transactions`"
DUP_LOSERS = f"`{CATALOG}`.`{SCHEMA}`.`duplicate_losers_transactions`"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print("Run timestamp          :", RUN_TS, f"(session time zone: {SESSION_TZ})")
print("Allowed payment methods:", ALLOWED_PAYMENT_METHODS)
print("Allowed currencies     :", ALLOWED_CURRENCIES)
print("Allowed txn statuses   :", ALLOWED_STATUSES)
print("Amount range           :", MIN_AMOUNT, "..", MAX_AMOUNT)
print("Timestamp range        :", MIN_TXN_DATE, ".. run timestamp")

## 2. Input checks
- `bronze_transactions` must be a single, non-empty snapshot.
- `silver_orders` must exist and be non-empty; otherwise **every** transaction would look like an orphan.

In [0]:
bronze_df = spark.table(BRONZE)
bronze_count = bronze_df.count()
batches = [r["_batch_id"] for r in bronze_df.select("_batch_id").distinct().collect()]

print(f"bronze_transactions rows : {bronze_count:,}")
print(f"batch id(s)              : {batches}")
assert bronze_count > 0, "bronze_transactions is empty - run 02_bronze_ingestion first."
assert len(batches) == 1, f"bronze_transactions holds {len(batches)} batches; expected exactly 1 snapshot."

silver_orders_count = spark.table(SILVER_ORDERS).count()
silver_orders_batches = [r["_batch_id"] for r in spark.table(SILVER_ORDERS).select("_batch_id").distinct().collect()]

print(f"silver_orders rows       : {silver_orders_count:,}  (batch: {silver_orders_batches})")
assert silver_orders_count > 0, "silver_orders is empty - run 03_silver_orders first."

## 3. Pre-flight: what values does Bronze actually contain?
Labels are shown after the same normalization Silver uses. Anything marked **WOULD BE QUARANTINED**
is not in your allowed list: either it is genuinely invalid (typo/junk) or your list is incomplete.

In [0]:
TRIM_RE = r"^\s+|\s+$"   # SQL-escaped regex for leading/trailing whitespace (spaces, tabs, newlines)


def clean(col):
    """Strip leading/trailing whitespace of any kind."""
    return f"regexp_replace({col}, '{TRIM_RE}', '')"


def norm_label(col, how):
    """Trim, turn blank into NULL, then lower/upper-case."""
    return f"{how}(NULLIF({clean(col)}, ''))"


def distribution(col, how, allowed):
    return spark.sql(f"""
        SELECT {norm_label(col, how)} AS {col}_normalized,
               COUNT(*) AS rows,
               CASE WHEN {norm_label(col, how)} IN ({sql_in_list(allowed)})
                    THEN 'allowed' ELSE 'WOULD BE QUARANTINED' END AS rule_result
        FROM {BRONZE}
        GROUP BY 1, 3
        ORDER BY rows DESC
        LIMIT 60
    """)

In [0]:
display(distribution("payment_method", "LOWER", ALLOWED_PAYMENT_METHODS))

In [0]:
display(distribution("currency", "UPPER", ALLOWED_CURRENCIES))

In [0]:
display(distribution("txn_status", "LOWER", ALLOWED_STATUSES))

In [0]:
# Optional helper: values that make up at least 0.5% of rows, as a ready-to-paste widget string.
# This is only a SUGGESTION. Review it: a frequent typo would be included, a rare valid value would not.
def suggest_list(col, how, min_share=0.005):
    rows = spark.sql(f"SELECT {norm_label(col, how)} AS v, COUNT(*) AS n FROM {BRONZE} GROUP BY 1").collect()
    total = sum(r["n"] for r in rows)
    vals = sorted(r["v"] for r in rows if r["v"] is not None and r["n"] / total >= min_share)
    return ",".join(vals)


print("Suggested allowed_payment_methods:", suggest_list("payment_method", "LOWER"))
print("Suggested allowed_currencies     :", suggest_list("currency", "UPPER"))
print("Suggested allowed_statuses       :", suggest_list("txn_status", "LOWER"))

In [0]:
# Observed ranges among values that parse (helps you set min/max amount and min_txn_date)
display(spark.sql(f"""
    SELECT
        MIN(try_cast({clean('amount')} AS DECIMAL(18,2)))   AS min_amount,
        MAX(try_cast({clean('amount')} AS DECIMAL(18,2)))   AS max_amount,
        MIN(try_cast({clean('txn_ts')} AS TIMESTAMP))       AS min_txn_ts,
        MAX(try_cast({clean('txn_ts')} AS TIMESTAMP))       AS max_txn_ts,
        SUM(CASE WHEN try_cast({clean('amount')} AS DECIMAL(18,2)) < 0 THEN 1 ELSE 0 END) AS negative_amount_rows
    FROM {BRONZE}
"""))

In [0]:
# Timestamp text shapes: tells you if the strict format rule below matches your file
display(spark.sql(f"""
    SELECT
        regexp_replace(regexp_replace({clean('txn_ts')}, '[0-9]', '9'), '[A-Za-z]', 'a') AS shape,
        COUNT(*) AS rows
    FROM {BRONZE}
    GROUP BY 1
    ORDER BY rows DESC
    LIMIT 20
"""))

## 4. Step A - clean text and parse types safely
`try_cast` yields NULL on bad input, and **every NULL is caught by a rule in step B**.

| Field | Cleaning / parsing |
|---|---|
| `txn_id`, `order_id` | strip whitespace; blank -> NULL; case is **kept** (same as orders, so the join is consistent) |
| `amount` | `DECIMAL(18,2)` (values with more than 2 decimals are rounded) |
| `txn_ts` | must look like `yyyy-MM-dd[ T]HH:mm:ss[.fraction][Z|+hh:mm]` AND be a real timestamp, then cast in the session time zone |
| `payment_method`, `txn_status` | strip whitespace, lowercase |
| `currency` | strip whitespace, uppercase |

In [0]:
TS_RE = "^[0-9]{4}-[0-9]{2}-[0-9]{2}[ T][0-9]{2}:[0-9]{2}:[0-9]{2}([.][0-9]+)?(Z|[+-][0-9]{2}:?[0-9]{2})?$"

# Layer 1: cleaned text (raw columns are kept untouched alongside, for quarantine evidence)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_p1 AS
SELECT
    txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,   -- raw strings
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    NULLIF({clean('txn_id')}, '')    AS txn_id_c,
    NULLIF({clean('order_id')}, '')  AS order_id_c,
    NULLIF({clean('amount')}, '')    AS amount_txt,
    NULLIF({clean('txn_ts')}, '')    AS ts_txt,
    {norm_label('payment_method', 'LOWER')}  AS payment_method_c,
    {norm_label('currency', 'UPPER')}        AS currency_c,
    {norm_label('txn_status', 'LOWER')}      AS status_c
FROM {BRONZE}
""")

# Layer 2: typed values + the order reference lookup.
# silver_orders.order_id is unique, so this LEFT JOIN cannot multiply rows.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_p2 AS
SELECT
    t.*,
    try_cast(t.amount_txt AS DECIMAL(18,2))                                         AS amount_p,
    CASE WHEN t.ts_txt RLIKE '{TS_RE}' THEN try_cast(t.ts_txt AS TIMESTAMP) END     AS txn_ts_p,
    (o.so_order_id IS NOT NULL)                                                     AS order_exists_in_silver
FROM txn_p1 t
LEFT JOIN (SELECT order_id AS so_order_id FROM {SILVER_ORDERS}) o
       ON t.order_id_c = o.so_order_id
""")

print("Temp views txn_p1 and txn_p2 created (no data written yet).")

## 5. Step B - validation matrix
Each failed rule adds a **reason code**; a row can have several.

| Field | Reason codes |
|---|---|
| `txn_id` | `txn_id_missing` |
| `order_id` | `order_id_missing`, `order_id_not_in_silver_orders` |
| `payment_method` | `payment_method_missing`, `payment_method_not_allowed` |
| `amount` | `amount_missing`, `amount_not_numeric`, `amount_below_min`, `amount_above_max` |
| `currency` | `currency_missing`, `currency_not_allowed` |
| `txn_status` | `txn_status_missing`, `txn_status_not_allowed` |
| `txn_ts` | `txn_ts_missing`, `txn_ts_invalid`, `txn_ts_future`, `txn_ts_too_old` |

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_validated AS
SELECT
    *,
    filter(
        array(
            CASE WHEN txn_id_c IS NULL THEN 'txn_id_missing' END,

            CASE WHEN order_id_c IS NULL                THEN 'order_id_missing'
                 WHEN NOT order_exists_in_silver        THEN 'order_id_not_in_silver_orders' END,

            CASE WHEN payment_method_c IS NULL          THEN 'payment_method_missing'
                 WHEN payment_method_c NOT IN ({sql_in_list(ALLOWED_PAYMENT_METHODS)}) THEN 'payment_method_not_allowed' END,

            CASE WHEN amount_txt IS NULL                THEN 'amount_missing'
                 WHEN amount_p IS NULL                  THEN 'amount_not_numeric'
                 WHEN amount_p < {MIN_AMOUNT}           THEN 'amount_below_min'
                 WHEN amount_p > {MAX_AMOUNT}           THEN 'amount_above_max' END,

            CASE WHEN currency_c IS NULL                THEN 'currency_missing'
                 WHEN currency_c NOT IN ({sql_in_list(ALLOWED_CURRENCIES)}) THEN 'currency_not_allowed' END,

            CASE WHEN status_c IS NULL                  THEN 'txn_status_missing'
                 WHEN status_c NOT IN ({sql_in_list(ALLOWED_STATUSES)}) THEN 'txn_status_not_allowed' END,

            CASE WHEN ts_txt IS NULL                    THEN 'txn_ts_missing'
                 WHEN txn_ts_p IS NULL                  THEN 'txn_ts_invalid'
                 WHEN txn_ts_p > TIMESTAMP'{RUN_TS}'    THEN 'txn_ts_future'
                 WHEN txn_ts_p < TIMESTAMP'{MIN_TXN_DATE} 00:00:00' THEN 'txn_ts_too_old' END
        ),
        x -> x IS NOT NULL
    ) AS reasons
FROM txn_p2
""")

# Only VALID rows compete for deduplication.
# WINNER RULE (my choice - document it): for the same txn_id among valid rows
#   1) latest txn_ts wins, 2) then smallest _row_hash, 3) then smallest _row_id  -> fully deterministic.
spark.sql("""
CREATE OR REPLACE TEMP VIEW txn_valid_ranked AS
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY txn_id_c ORDER BY txn_ts_p DESC, _row_hash ASC, _row_id ASC) AS dup_rank,
    FIRST_VALUE(_row_id) OVER (PARTITION BY txn_id_c ORDER BY txn_ts_p DESC, _row_hash ASC, _row_id ASC) AS winner_row_id,
    COUNT(*) OVER (PARTITION BY txn_id_c) AS group_size
FROM txn_validated
WHERE size(reasons) = 0
""")

print("Temp views txn_validated and txn_valid_ranked created.")

### Dry run: how many rows go where? (nothing written yet)

In [0]:
preview = spark.sql("""
    SELECT
        COUNT(*)                                            AS bronze_rows,
        SUM(CASE WHEN size(reasons) > 0 THEN 1 ELSE 0 END)  AS would_quarantine,
        SUM(CASE WHEN size(reasons) = 0 THEN 1 ELSE 0 END)  AS valid_rows
    FROM txn_validated
""").collect()[0]

quarantine_pct = preview["would_quarantine"] / preview["bronze_rows"] * 100
print(f"Bronze rows      : {preview['bronze_rows']:,}")
print(f"Would quarantine : {preview['would_quarantine']:,}  ({quarantine_pct:.2f}%)")
print(f"Valid rows       : {preview['valid_rows']:,}")

if quarantine_pct > 50:
    print("\nWARNING: more than half of the rows would be quarantined. "
          "Check the allowed lists/limits against the pre-flight output (section 3), "
          "and confirm 03_silver_orders accepted a sensible number of orders.")

In [0]:
display(spark.sql("""
    SELECT reason, COUNT(*) AS rows
    FROM txn_validated
    LATERAL VIEW explode(reasons) r AS reason
    GROUP BY reason
    ORDER BY rows DESC
"""))

## 6. Step C - write the three outcome tables
All three come from the same Bronze snapshot and the same `silver_orders` state.
Audit identifiers are kept in every table.

In [0]:
# 6a. ACCEPTED: one deterministic winner per txn_id
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}
USING DELTA
COMMENT 'Silver: valid, typed, deduplicated transactions. One row per txn_id. order_id exists in silver_orders.'
AS
SELECT
    txn_id_c            AS txn_id,
    order_id_c          AS order_id,
    payment_method_c    AS payment_method,
    amount_p            AS amount,
    currency_c          AS currency,
    status_c            AS txn_status,
    txn_ts_p            AS txn_ts,
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    CAST(group_size AS INT)   AS _duplicate_group_size,
    current_timestamp()       AS _silver_processed_at
FROM txn_valid_ranked
WHERE dup_rank = 1
""")
print("silver_transactions written")

In [0]:
# 6b. QUARANTINE: original raw text + every reason the row failed
spark.sql(f"""
CREATE OR REPLACE TABLE {QUARANTINE}
USING DELTA
COMMENT 'Quarantine: transactions that failed at least one validation rule (incl. orphan order_id). Raw values kept as text.'
AS
SELECT
    txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,   -- raw values
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    concat_ws('; ', reasons)      AS rejection_reason,
    reasons                       AS rejection_reasons,
    CAST(size(reasons) AS INT)    AS rejection_count,
    current_timestamp()           AS _silver_processed_at
FROM txn_validated
WHERE size(reasons) > 0
""")
print("quarantine_transactions written")

In [0]:
# 6c. DUPLICATE LOSERS: valid rows that lost to the winner of the same txn_id
spark.sql(f"""
CREATE OR REPLACE TABLE {DUP_LOSERS}
USING DELTA
COMMENT 'Valid transactions that lost deterministic winner selection for their txn_id. Linked to the winner by winner_row_id.'
AS
SELECT
    txn_id_c            AS txn_id,
    order_id_c          AS order_id,
    payment_method_c    AS payment_method,
    amount_p            AS amount,
    currency_c          AS currency,
    status_c            AS txn_status,
    txn_ts_p            AS txn_ts,
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    winner_row_id,
    CAST(dup_rank AS INT)     AS duplicate_rank,
    CAST(group_size AS INT)   AS _duplicate_group_size,
    'latest txn_ts, then smallest _row_hash, then smallest _row_id' AS winner_rule,
    current_timestamp()       AS _silver_processed_at
FROM txn_valid_ranked
WHERE dup_rank > 1
""")
print("duplicate_losers_transactions written")

## 7. Validation: every Bronze row has exactly one outcome
Totals are not enough, so this also proves **no overlap** and **nothing missing** at row level using `_row_id`,
and that every accepted transaction really points to an accepted order.

In [0]:
bronze_n = spark.table(BRONZE).count()
silver_n = spark.table(SILVER).count()
quar_n = spark.table(QUARANTINE).count()
dup_n = spark.table(DUP_LOSERS).count()

print(f"Bronze rows            : {bronze_n:,}")
print(f"  accepted (silver)    : {silver_n:,}")
print(f"  quarantined          : {quar_n:,}")
print(f"  duplicate losers     : {dup_n:,}")
print(f"  accepted+quar+losers : {silver_n + quar_n + dup_n:,}")

display(spark.createDataFrame(
    [("bronze_transactions", bronze_n), ("silver_transactions (accepted)", silver_n),
     ("quarantine_transactions", quar_n), ("duplicate_losers_transactions", dup_n),
     ("accepted + quarantine + losers", silver_n + quar_n + dup_n)],
    "table STRING, rows LONG"
))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_outcomes AS
SELECT _row_id, 'accepted'         AS outcome FROM {SILVER}
UNION ALL
SELECT _row_id, 'quarantine'       AS outcome FROM {QUARANTINE}
UNION ALL
SELECT _row_id, 'duplicate_loser'  AS outcome FROM {DUP_LOSERS}
""")

outcome_rows = spark.sql("SELECT COUNT(*) AS n, COUNT(DISTINCT _row_id) AS d FROM txn_outcomes").collect()[0]

missing_from_outcomes = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {BRONZE} b
    LEFT ANTI JOIN txn_outcomes o ON b._row_id = o._row_id
""").collect()[0]["n"]

unknown_in_outcomes = spark.sql(f"""
    SELECT COUNT(*) AS n FROM txn_outcomes o
    LEFT ANTI JOIN {BRONZE} b ON b._row_id = o._row_id
""").collect()[0]["n"]

dup_txn_ids_in_silver = spark.sql(f"""
    SELECT COUNT(*) - COUNT(DISTINCT txn_id) AS n FROM {SILVER}
""").collect()[0]["n"]

losers_without_winner = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {DUP_LOSERS} l
    LEFT ANTI JOIN {SILVER} s ON l.winner_row_id = s._row_id
""").collect()[0]["n"]

quarantine_without_reason = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {QUARANTINE}
    WHERE rejection_reason IS NULL OR trim(rejection_reason) = ''
""").collect()[0]["n"]

# Referential integrity: every accepted transaction must point to an accepted order
silver_orphans = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {SILVER} t
    LEFT ANTI JOIN {SILVER_ORDERS} o ON t.order_id = o.order_id
""").collect()[0]["n"]

# Spot-check that accepted rows really satisfy the rules
rule_violations_in_silver = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {SILVER}
    WHERE txn_id IS NULL OR order_id IS NULL
       OR amount IS NULL OR amount < {MIN_AMOUNT} OR amount > {MAX_AMOUNT}
       OR txn_ts IS NULL OR txn_ts > TIMESTAMP'{RUN_TS}' OR txn_ts < TIMESTAMP'{MIN_TXN_DATE} 00:00:00'
       OR payment_method NOT IN ({sql_in_list(ALLOWED_PAYMENT_METHODS)})
       OR currency NOT IN ({sql_in_list(ALLOWED_CURRENCIES)})
       OR txn_status NOT IN ({sql_in_list(ALLOWED_STATUSES)})
""").collect()[0]["n"]

checks = {
    "bronze = accepted + quarantine + losers": bronze_n == silver_n + quar_n + dup_n,
    "no Bronze row is missing from outcomes": missing_from_outcomes == 0,
    "no row appears in more than one outcome (distinct _row_id = outcome rows)": outcome_rows["n"] == outcome_rows["d"],
    "no outcome row is unknown to Bronze": unknown_in_outcomes == 0,
    "silver_transactions has unique txn_id": dup_txn_ids_in_silver == 0,
    "every duplicate loser points to an accepted winner": losers_without_winner == 0,
    "every quarantined row has a rejection_reason": quarantine_without_reason == 0,
    "every accepted transaction references an accepted order": silver_orphans == 0,
    "accepted rows satisfy all documented rules": rule_violations_in_silver == 0,
}

print("=" * 70)
for label, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
print("=" * 70)
print("OVERALL:", "SILVER TRANSACTIONS VALID" if all(checks.values()) else "SILVER TRANSACTIONS CHECKS FAILED")

assert all(checks.values()), "Silver transactions validation failed - see output above."

## 8. Orphan diagnostics
Why were transactions rejected for `order_id_not_in_silver_orders`? This separates
*the order exists in Bronze but was rejected* from *the order id never existed at all*.

In [0]:
display(spark.sql(f"""
    WITH orphans AS (
        SELECT NULLIF({clean('order_id')}, '') AS oid
        FROM {QUARANTINE}
        WHERE array_contains(rejection_reasons, 'order_id_not_in_silver_orders')
    ),
    q_orders AS (SELECT DISTINCT NULLIF({clean('order_id')}, '') AS oid FROM `{CATALOG}`.`{SCHEMA}`.`quarantine_orders`)
    SELECT
        CASE WHEN q.oid IS NOT NULL THEN 'order exists but was quarantined in Silver'
             ELSE 'order_id not found in any orders outcome' END AS orphan_cause,
        COUNT(*) AS transactions
    FROM orphans o
    LEFT JOIN q_orders q ON o.oid = q.oid
    GROUP BY 1
    ORDER BY transactions DESC
"""))

## 9. Evidence for `evidence/04_silver/`

In [0]:
# Rejection reasons (a row with several reasons counts once per reason, so this can exceed the quarantine row count)
display(spark.sql(f"""
    SELECT reason AS rejection_reason, COUNT(*) AS rows_with_this_reason
    FROM {QUARANTINE}
    LATERAL VIEW explode(rejection_reasons) r AS reason
    GROUP BY reason
    ORDER BY rows_with_this_reason DESC
"""))

In [0]:
# Sample quarantined rows (raw values + reasons)
display(spark.sql(f"""
    SELECT txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts, rejection_reason, _row_id
    FROM {QUARANTINE}
    ORDER BY rejection_count DESC
    LIMIT 20
"""))

In [0]:
# Sample orphan-order quarantines specifically (the guide asks for orphan evidence)
display(spark.sql(f"""
    SELECT txn_id, order_id, amount, txn_status, rejection_reason, _row_id
    FROM {QUARANTINE}
    WHERE array_contains(rejection_reasons, 'order_id_not_in_silver_orders')
    LIMIT 20
"""))

In [0]:
# Duplicate losers next to their winner, to show the deterministic rule at work
display(spark.sql(f"""
    SELECT l.txn_id,
           l.duplicate_rank, l.txn_ts AS loser_txn_ts, w.txn_ts AS winner_txn_ts,
           l._row_id AS loser_row_id, w._row_id AS winner_row_id, l.winner_rule
    FROM {DUP_LOSERS} l
    JOIN {SILVER} w ON l.winner_row_id = w._row_id
    ORDER BY l.txn_id, l.duplicate_rank
    LIMIT 20
"""))

In [0]:
display(spark.sql(f"DESCRIBE TABLE {SILVER}"))

In [0]:
display(spark.table(SILVER).limit(10))

## 10. README text: the rules you applied
Built from the widget values actually used in this run.

In [0]:
print(f"""
## Silver transactions: validation, parsing, reference and deduplication rules

Run timestamp used for the future-timestamp rule: {RUN_TS} (session time zone {SESSION_TZ})
Bronze batch: {batches[0]}
Reference table: silver_orders ({silver_orders_count:,} accepted orders)

**Precedence:** validate first (including the order reference), then deduplicate among valid records.

| Field | Parsing / normalization | Rule (my chosen limits, not assignment-provided) | Failure outcome |
|---|---|---|---|
| txn_id | trim, blank -> NULL | not blank | quarantine |
| order_id | trim, blank -> NULL | not blank AND exists in accepted silver_orders | quarantine (orphan) |
| payment_method | trim + lowercase | in {ALLOWED_PAYMENT_METHODS} | quarantine |
| amount | DECIMAL(18,2) | numeric, {MIN_AMOUNT} .. {MAX_AMOUNT} | quarantine |
| currency | trim + uppercase | in {ALLOWED_CURRENCIES} | quarantine |
| txn_status | trim + lowercase | in {ALLOWED_STATUSES} | quarantine |
| txn_ts | strict ISO-style text -> TIMESTAMP | parses; >= {MIN_TXN_DATE}; <= run timestamp | quarantine |
| duplicate txn_id | among VALID rows only | winner = latest txn_ts, then smallest _row_hash, then smallest _row_id | duplicate_losers_transactions |

**Order-reference rule:** a transaction is valid only if its order_id exists in the accepted silver_orders.
Orders that were quarantined make their transactions orphans. An order id with duplicate rows is valid if one row won.

**Outcome counts (this run):** Bronze {bronze_n:,} = accepted {silver_n:,} + quarantined {quar_n:,} + duplicate losers {dup_n:,}

**Rerun behavior:** tables are rebuilt with CREATE OR REPLACE TABLE, so reruns do not duplicate rows.
**Dependency:** run 03_silver_orders before this notebook; if orders are rebuilt, rerun this notebook too.
""")

## Done
Save for `evidence/04_silver/`: the outcome-count table, the PASS summary, the rejection-reason breakdown,
the orphan examples and a duplicate winner/loser sample.

**Next:** `05_reconciliation.py` (repeats the invariant for both datasets and writes the SQL assertions).